# 3D Navier-Stokes in `dynestyx`

In [7]:
import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
import functools
import os

import jax.numpy as jnp
import jax.random as jr
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML

import exponax
import exponax as ex


import dynestyx as dsx
from dynestyx import (
    DiracInitialCondition,
    DiracObservation,
    GaussianObservation,
    DiracStateEvolution,
    DynamicalModel,
    Layout,
)
import math


# Utilities

In [9]:
@functools.cache
def _fourier_operators(n):
    kf = 2 * jnp.pi * jnp.fft.fftfreq(n, d=domain_extent / n)
    kr = 2 * jnp.pi * jnp.fft.rfftfreq(n, d=domain_extent / n)
    K = jnp.stack(jnp.meshgrid(kf, kf, kr, indexing="ij"))
    return K


def leray_project(field):
    """Project a (3, N, N, N) velocity field onto divergence-free fields."""
    n = field.shape[-1]
    K = _fourier_operators(n)
    field_hat = jnp.fft.rfftn(field, axes=(1, 2, 3))
    k_squared = jnp.where((K**2).sum(axis=0) == 0, 1.0, (K**2).sum(axis=0))
    field_hat = field_hat - K * (K * field_hat).sum(axis=0) / k_squared
    return jnp.fft.irfftn(field_hat, s=(n,) * 3, axes=(1, 2, 3))


def vorticity_magnitude(field):
    """Compute |curl u| from a (3, N, N, N) velocity field."""
    K = _fourier_operators(field.shape[-1])
    field_hat = jnp.fft.rfftn(field, axes=(1, 2, 3))
    omega = jnp.stack([
        jnp.fft.irfftn(
            1j * (K[b] * field_hat[c] - K[c] * field_hat[b]),
            s=field.shape[1:],
            axes=(0, 1, 2),
        )
        for b, c in [(1, 2), (2, 0), (0, 1)]
    ])
    return jnp.sqrt((omega**2).sum(axis=0))



# Defining the PDE dynamics

A forward simulation of the three-dimensional incompressible Navier-Stokes equations as a
single `dynestyx.DynamicalModel`. In velocity formulation,

$$
\partial_t \mathbf{u} = \nu \Delta \mathbf{u} + \lambda \mathbf{u}
    + \mathcal{P}\big(\mathbf{u} \times \boldsymbol{\omega}\big) + \mathbf{f},
\qquad
\boldsymbol{\omega} = \nabla \times \mathbf{u},
\qquad
\nabla \cdot \mathbf{u} = 0 .
$$

$\nu$ is the viscosity and $\lambda$ a weak drag. The nonlinearity is in rotational form,
$(\mathbf{u}\cdot\nabla)\mathbf{u} = \nabla(|\mathbf{u}|^2/2) + \boldsymbol{\omega}\times\mathbf{u}$,
so the Leray projection $\mathcal{P}$ removes the
pressure term and enforces incompressibility in one step. The Kolmogorov forcing

$$
\mathbf{f} = (\gamma \sin(k \tfrac{2\pi}{L} y), 0, 0)
$$

injects energy so the turbulence is
sustained.

The observations are a coarsed grained version of the PDE, polluted by Gaussian noise:

$$
y_t = G * u_t + \varepsilon_t
$$



We use [`exponax`](https://fkoehler.site/exponax/) spectral solver. In this case, the dynamics are deterministic: the transition returns a `Delta` distribution.

In [ ]:
domain_extent = float(2 * jnp.pi) # periodic box (0, L)^3
dt_obs = 0.25                     # the dynestyx model time step

num_points = 64                   # grid points per axis. Warning: increasing this means also increasing the number of solver substeps.
solver_substeps = 10              # how many internal solver steps per observation step. Heuristic: N = 128 needs 25 substeps, N = 64 needs 10 substeps, N = 32 needs 5 substeps. This is to ensure stability of the solver.

# Parameters of the PDE 
viscosity = 0.005                  # nu
drag = -0.1                        # lambda
injection_mode = 4                 # forcing wavenumber k


axis = jnp.linspace(0, domain_extent, num_points, endpoint=False)

# We start from a random initial condition projected onto divergence-free fields.
random_ic = ex.ic.RandomTruncatedFourierSeries(3, cutoff=2, std_one=True)
u_init = leray_project(
    jnp.concatenate([random_ic(num_points, key=jr.key(seed)) for seed in range(3)])
)

In [ ]:
# Defining the PDE stepper
def make_stepper(num_points, dt_obs):
    dt_solver = dt_obs / solver_substeps 
    pde_stepper = exponax.RepeatedStepper(
        exponax.stepper.KolmogorovFlowVelocity(
            num_spatial_dims=3,
            domain_extent=domain_extent,
            num_points=num_points,
            dt=dt_solver,
            diffusivity=viscosity,
            drag=drag,
            injection_mode=injection_mode,
            order=4,  
        ),
        solver_substeps,
    )
    def stepper(state, u, t_now, t_next):
        return pde_stepper(state)
    return stepper

step = make_stepper(num_points, dt_obs)

# Define the observation function: a downsampling operator.
def observation_function(state, u, t):
    return exponax.map_between_resolutions(state, num_points // 4)

state_layout = Layout.from_example(u_init)
observation_layout = Layout.from_example(observation_function(u_init, None, 0.0))

In [ ]:
initial_condition = DiracInitialCondition(u_init, state_layout=state_layout)
evolution = DiracStateEvolution(F=step, state_layout=state_layout)
observation_model = GaussianObservation( observation_function, state_layout=state_layout, 
                                        observation_layout=observation_layout,
                                        cov = 0.01)
dynamics = DynamicalModel(
        initial_condition=initial_condition,
        state_evolution=evolution,
        observation_model=observation_model,
        state_layout=state_layout,
        observation_layout=observation_layout,
    )

In [29]:
T = 50.0
times = jnp.arange(0.0, T, step=dt_obs)

result = dsx.simulate(
    dynamics,
    rng_key=jr.key(13),
    predict_times=times,
)
states = result.states[0]
assert jnp.isfinite(states).all()

In [ ]:
t_start = 5.0
states = result.states[0][int(t_start/dt_obs):]  # discard the first 5 seconds of the simulation (for visualization only, so we only view the steady state).
observations = result.observations[0][int(t_start/dt_obs):] 

# Vortex tubes

We compute the vorticity magnitude and plot the strongest cells, revealing the
intermittent tubes and sheets produced by the turbulent flow.

In [32]:
anim_stride = 2   # frames to animate = n_steps // anim_stride
animation_points = 128  # visualization grid only; the PDE still runs at num_points

# Fourier zero-padding evaluates the same band-limited velocity field on a finer grid.
# It introduces no new resolved modes; it only makes the visualization less blocky.
wmag = np.stack([
    np.asarray(
        vorticity_magnitude(ex.map_between_resolutions(s, animation_points)),
        dtype=np.float32,
    )
    for s in states[::anim_stride]
])


threshold_pct = 0.9
threshold = float(np.percentile(wmag, threshold_pct))    # which cells are drawn at all
colour_min = threshold                                   # bottom of the colour scale
colour_max = float(np.percentile(wmag, 97.0))            # top; anything above clips

coords = np.linspace(0.0, domain_extent, animation_points, endpoint=False)
n_cells = animation_points**3
print(f"|omega| > {threshold:.2f} (p{threshold_pct:.2f}): "
      f"~{int((wmag[0] > threshold).sum()):,} of {n_cells:,} cells per frame;  "
      f"colour scale [{colour_min:.2f}, {colour_max:.2f}]")

fig = plt.figure(figsize=(4.6, 4.6), dpi=76, facecolor="white")
ax = fig.add_axes([0.0, 0.0, 1.0, 1.0], projection="3d", facecolor="white")


def draw(frame):
    ax.clear()
    ax.set_facecolor("white")
    wm = wmag[frame]
    i, j, k = np.nonzero(wm > threshold)
    ax.scatter(coords[i], coords[j], coords[k], c=wm[i, j, k], cmap="turbo",
               vmin=colour_min, vmax=colour_max, s=9, alpha=0.6,
               linewidths=0, depthshade=False)
    ax.set_xlim(0, domain_extent); ax.set_ylim(0, domain_extent)
    ax.set_zlim(0, domain_extent)
    ax.set_box_aspect((1, 1, 1), zoom=1.22)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    for a in (ax.xaxis, ax.yaxis, ax.zaxis):
        a.pane.set_facecolor("white"); a.pane.set_edgecolor("0.25")
        a.line.set_color("0.25")
    ax.view_init(elev=20, azim=30 + 1.5 * frame)
    return []


draw(0)
ani = animation.FuncAnimation(fig, draw, frames=wmag.shape[0], interval=110)
plt.close(fig)

anim_path = "vortex_tubes.mp4"
ani.save(anim_path, writer="ffmpeg", fps=9, dpi=76,
         extra_args=["-pix_fmt", "yuv420p", "-crf", "28"],
         savefig_kwargs={"facecolor": "white"})
print(f"wrote {anim_path}: {os.path.getsize(anim_path) / 1e6:.2f} MB")

# Reference the file rather than embedding it, so the notebook stays small.
HTML(f'<video src="{anim_path}" controls loop width="460"></video>')

|omega| > 0.46 (p0.90): ~2,085,390 of 2,097,152 cells per frame;  colour scale [0.46, 7.81]
wrote vortex_tubes.mp4: 1.36 MB


In [44]:
# Fourier zero-padding evaluates the same band-limited velocity field on a finer grid.
# It introduces no new resolved modes; it only makes the visualization less blocky.
wmag = np.stack([
    np.asarray(
        vorticity_magnitude(ex.map_between_resolutions(s, animation_points)),
        dtype=np.float32,
    )
    for s in states[::anim_stride]
])

# Compare the full state with its coarse volumetric observation. The observation is
# Fourier-interpolated back to the animation grid for display only.
observations = result.observations[0]
observation_wmag = np.stack([
    np.asarray(
        vorticity_magnitude(
            ex.map_between_resolutions(observation, animation_points)
        ),
        dtype=np.float32,
    )
    for observation in observations[::anim_stride]
])

comparison_fields = (wmag, observation_wmag)
comparison_titles = (r"$\mathrm{True\ state}$", r"$\mathrm{Observation}$")

fig = plt.figure(figsize=(9.2, 4.8), dpi=76, facecolor="white")
comparison_axes = [
    fig.add_subplot(1, 2, index + 1, projection="3d", facecolor="white")
    for index in range(2)
]
# Reserve a generous top margin so the titles do not touch or clip against the cubes.
fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.84, wspace=0.02)


def draw_comparison(frame):
    for ax, field, title in zip(comparison_axes, comparison_fields, comparison_titles):
        ax.clear()
        ax.set_facecolor("white")
        wm = field[frame]
        i, j, k = np.nonzero(wm > threshold)
        ax.scatter(
            coords[i], coords[j], coords[k],
            c=wm[i, j, k], cmap="turbo",
            vmin=colour_min, vmax=colour_max, s=9, alpha=0.6,
            linewidths=0, depthshade=False,
        )
        ax.set_xlim(0, domain_extent); ax.set_ylim(0, domain_extent)
        ax.set_zlim(0, domain_extent)
        ax.set_box_aspect((1, 1, 1), zoom=1.05)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
        for axis_3d in (ax.xaxis, ax.yaxis, ax.zaxis):
            axis_3d.pane.set_facecolor("white")
            axis_3d.pane.set_edgecolor("0.25")
            axis_3d.line.set_color("0.25")
        ax.view_init(elev=20, azim=30 + 1.5 * frame)
        ax.set_title(title, fontsize=18, pad=14, math_fontfamily="cm")
    return []


draw_comparison(0)
comparison_ani = animation.FuncAnimation(
    fig, draw_comparison, frames=wmag.shape[0], interval=110
)
plt.close(fig)

comparison_path = "vortex_tubes_observation.mp4"
comparison_ani.save(
    comparison_path, writer="ffmpeg", fps=9, dpi=76,
    extra_args=["-pix_fmt", "yuv420p", "-crf", "28"],
    savefig_kwargs={"facecolor": "white"},
)
print(f"wrote {comparison_path}: {os.path.getsize(comparison_path) / 1e6:.2f} MB")
HTML(f'<video src="{comparison_path}" controls loop width="920"></video>')

wrote vortex_tubes_observation.mp4: 1.63 MB
